# Study 2

Compared point, moderate-range, and very wide-range delivery estimates to test whether making a range too wide reduces satisfaction.

In [1]:
# Import packages and locate the data folder.
from pathlib import Path
import os
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import binomtest, t
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/raw/Study 1.xlsx').exists())
os.environ['MPLCONFIGDIR'] = str(ROOT / '.cache/matplotlib')
ITEMS = ['sb_like', 'sb_info', 'sb_useful', 'sb_accurate', 'sb_trust']
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 4)

In [2]:
# Load data, apply exclusions, and calculate satisfaction.
d = pd.read_excel(ROOT / 'data/raw/Study 2.xlsx')
raw_n = len(d)

d = d.loc[(d.ExcludeDuplicateIP != 1) & (d.ExcludeDuplicateID != 1)].copy()
assert len(d) == 599
d['satisfaction'] = d[ITEMS].mean(axis=1)
rows, descriptions = [], []
display(pd.DataFrame([{'raw_n':raw_n, 'replicated_n':len(d), 'paper_n':599, 'match':len(d)==599}]))

,raw_n,replicated_n,paper_n,match
0,600,599,599,True


In [3]:
# Define helpers for regressions and paper comparisons.
def record(study, label, fit, term, target_b=None, target_se=None, sign=1):
    ci = np.sort(np.asarray(fit.conf_int().loc[term]) * sign)
    b, se = (sign * fit.params[term], fit.bse[term])
    row = {'study': study, 'analysis': label, 'n': int(fit.nobs), 'b': b, 'se': se, 'p': fit.pvalues[term], 'ci_low': ci[0], 'ci_high': ci[1], 'published_b': target_b, 'published_se': target_se, 'b_matches_2dp': None if target_b is None else abs(b - target_b) <= 0.0050001, 'se_matches_2dp': None if target_se is None else abs(se - target_se) <= 0.0050001}
    rows.append(row)

def ols(d, y, terms):
    return sm.OLS(d[y], sm.add_constant(d[terms].astype(float)), missing='drop').fit()

def contrast(d, study, labels, treatment, reference, target_b, target_se, label=None, covariates=()):
    frame = d.copy()
    for name, code in labels.items():
        frame[name] = (frame.Condition == code).astype(float)
    terms = [name for name in labels if name != reference] + list(covariates)
    record(study, label or f'{treatment} - {reference}', ols(frame, 'satisfaction', terms), treatment, target_b, target_se)

In [4]:
# Estimate satisfaction differences between point, moderate-range, and wide-range estimates.
labels = {'point': 1, 'moderate': 2, 'wide': 3}
for a,b,beta,se in [('moderate', 'point', 0.68, 0.13), ('wide', 'moderate', -0.99, 0.13), ('wide', 'point', -0.3, 0.14)]:
    contrast(d,2,labels,a,b,beta,se)
published = {1: (4.62, 1.41), 2: (5.31, 1.11), 3: (4.32, 1.5)}

In [5]:
# Display estimates beside the paper and check agreement at reported precision.
results = pd.DataFrame(rows)
checked = results.published_b.notna()
assert results.loc[checked, 'b_matches_2dp'].all()
assert results.loc[checked, 'se_matches_2dp'].all()
display(results[['analysis','n','b','published_b','se','published_se','p','b_matches_2dp','se_matches_2dp']])

,analysis,n,b,published_b,se,published_se,p,b_matches_2dp,se_matches_2dp
0,moderate - point,599,0.6833,0.68,0.1345,0.13,5.0633e-07,True,True
1,wide - moderate,599,-0.9860,-0.99,0.1343,0.13,7.0727e-13,True,True
2,wide - point,599,-0.3027,-0.30,0.1360,0.14,2.6438e-02,True,True


In [6]:
# Compare group means and standard deviations with the published table.
table = d.groupby('Condition').satisfaction.agg(n='count',mean='mean',sd='std')
table['condition'] = table.index.map({v:k for k,v in labels.items()})
table['paper_mean'] = [published[c][0] for c in table.index]
table['paper_sd'] = [published[c][1] for c in table.index]
table['match'] = ((table['mean']-table.paper_mean).abs()<=.0050001)&((table.sd-table.paper_sd).abs()<=.0050001)
assert table['match'].all()
display(table)

,n,mean,sd,condition,paper_mean,paper_sd,match
Condition,,,,,,,
1,196,4.6235,1.4144,point,4.62,1.41,True
2,206,5.3068,1.1129,moderate,5.31,1.11,True
3,197,4.3208,1.4970,wide,4.32,1.50,True
